# CERES-Maize Simulation Example

This notebook demonstrates how to run a CERES-Maize simulation using the DSSAT Python package.
We will use the Gainesville, FL 1982 dataset (UFGA8201) — a classic DSSAT validation dataset.

**What this notebook covers:**
1. Loading an experiment from JSON
2. Running the simulation
3. Inspecting end-of-season summary results
4. Plotting daily outputs (phenology, LAI, biomass, soil water, stress)
5. Converting a legacy `.WTH` weather file to JSON using the built-in converters

## 1  Setup

In [ ]:
from pathlib import Path
import dssat as _dssat_pkg

# Derive project root from the installed package location (works in all execution contexts)
project_root = Path(_dssat_pkg.__file__).parent.parent
TESTS_DIR       = project_root / "tests"
EXPERIMENT_JSON = TESTS_DIR / "example_experiment.json"
WEATHER_JSON    = TESTS_DIR / "example_weather.json"

print("Project root :", project_root)
print("Experiment   :", EXPERIMENT_JSON.exists(), EXPERIMENT_JSON)
print("Weather      :", WEATHER_JSON.exists(), WEATHER_JSON)


In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

%matplotlib inline
plt.rcParams.update({"figure.dpi": 120, "font.size": 10})

## 2  Inspect the Experiment JSON

The experiment is defined entirely in a single JSON file.  
The soil profile is stored inline; weather comes from a companion JSON file.

In [ ]:
with open(EXPERIMENT_JSON) as f:
    exp = json.load(f)

print(f"Experiment ID : {exp['experiment_id']}")
print(f"Description   : {exp['description']}")
print(f"Crop model    : {exp['crop']['model']}")
print(f"Cultivar      : {exp['crop']['cultivar']['id']} – {exp['crop']['cultivar']['name']}")
print()
print("Planting:")
for k, v in exp["planting"].items():
    print(f"  {k:12s}: {v}")
print()
print("Cultivar coefficients:")
cv = exp["crop"]["cultivar"]
for k in ("p1", "p2", "p5", "g2", "g3", "phint"):
    print(f"  {k:6s}: {cv[k]}")
print()
print("Fertilizer events:")
for fe in exp.get("fertilizer", []):
    print(f"  YRDOY {fe['date']}  {fe['amount']} kg/ha at {fe['n_pct']}% N")

In [ ]:
# Quick look at the soil profile
layers = exp["soil"]["inline"]["layers"]
soil_df = pd.DataFrame(layers)
print("Soil profile:", exp["soil"]["inline"]["description"])
soil_df

## 3  Run the Simulation

`Simulation.from_json()` reads the experiment JSON, resolves the weather path relative to the
JSON file, and wires together all sub-models.

In [ ]:
from dssat import Simulation

sim = Simulation.from_json(EXPERIMENT_JSON)
results = sim.run()

print("Simulation complete.")
print("Keys returned:", list(results.keys()))

## 4  End-of-Season Summary

In [ ]:
summary = results["summary"]

print("=" * 40)
print("  CERES-Maize End-of-Season Summary")
print("=" * 40)
for key, val in summary.items():
    label = key.replace("_", " ").title()
    if isinstance(val, float):
        print(f"  {label:<30s}: {val:.2f}")
    else:
        print(f"  {label:<30s}: {val}")

## 5  Daily Output DataFrame

In [ ]:
daily = results["daily"]
print(f"{len(daily)} daily records, columns: {list(daily.columns)}")
daily.head(10)

In [ ]:
# Convenience: add calendar day-of-year relative to planting (DAS = Days After Sowing)
daily["das"] = range(len(daily))
daily.tail(10)

## 6  Plots

### 6.1  Phenology and Crop Growth

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(10, 9), sharex=True)

das = daily["das"]

# — Growth stage
ax = axes[0]
ax.step(das, daily["istage"], where="post", color="purple", linewidth=1.5)
stage_labels = {
    7: "Sowing", 8: "Germination", 9: "Emergence",
    1: "End Juvenile", 2: "Floral Init.", 3: "Silking",
    4: "Grain Fill", 5: "Maturity", 6: "Harvest",
}
for stage, label in stage_labels.items():
    first = daily.loc[daily["istage"] == stage, "das"]
    if not first.empty:
        ax.axvline(first.iloc[0], color="grey", linestyle=":", alpha=0.6)
        ax.text(first.iloc[0] + 0.3, stage + 0.1, label, fontsize=7, color="grey")
ax.set_ylabel("Growth Stage (istage)")
ax.set_yticks(list(range(1, 10)))
ax.set_title("CERES-Maize — Gainesville FL 1982 (UFGA8201)")

# — LAI
ax = axes[1]
ax.fill_between(das, daily["lai"], alpha=0.35, color="green")
ax.plot(das, daily["lai"], color="green", linewidth=1.2)
ax.set_ylabel("LAI (m² m⁻²)")

# — Biomass and grain yield
ax = axes[2]
ax.plot(das, daily["biomas_g_m2"] / 100, color="saddlebrown", label="Total biomass (g m⁻² ÷ 100)")
ax.plot(das, daily["yield_kg_ha"] / 100, color="goldenrod",    linestyle="--", label="Grain yield (kg ha⁻¹ ÷ 100)")
ax.set_ylabel("Scaled biomass / yield")
ax.set_xlabel("Days After Sowing")
ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig("ceres_maize_growth.png", bbox_inches="tight")
plt.show()
print("Figure saved: ceres_maize_growth.png")

### 6.2  Soil Water and Stress

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(10, 8), sharex=True)

# — Daily weather
ax = axes[0]
ax2 = ax.twinx()
ax.bar(das, daily["rain"], color="royalblue", alpha=0.6, width=1, label="Rain (mm)")
ax2.plot(das, daily["srad"], color="orange", linewidth=1, label="Solar rad. (MJ m⁻²)")
ax.set_ylabel("Rainfall (mm)")
ax2.set_ylabel("SRAD (MJ m⁻²)")
lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax.legend(lines1 + lines2, labels1 + labels2, fontsize=8, loc="upper right")
ax.set_title("Soil Water Balance and Crop Stress")

# — Soil water
ax = axes[1]
ax.plot(das, daily["tsw_cm"], color="steelblue", label="Total soil water (cm)")
ax.fill_between(das, daily["tsw_cm"], alpha=0.2, color="steelblue")
ax.set_ylabel("Total soil water (cm)")
ax.legend(fontsize=8)

# — Stress factors
ax = axes[2]
ax.plot(das, daily["swfac"],  color="red",    linewidth=1.2, label="Water stress (swfac)")
ax.plot(das, daily["nstres"], color="darkorange", linewidth=1.2, linestyle="--", label="N stress (nstres)")
ax.axhline(1.0, color="black", linewidth=0.6, linestyle=":")
ax.set_ylim(0, 1.1)
ax.set_ylabel("Stress factor (0=max, 1=none)")
ax.set_xlabel("Days After Sowing")
ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig("ceres_maize_water_stress.png", bbox_inches="tight")
plt.show()
print("Figure saved: ceres_maize_water_stress.png")

### 6.3  Temperature and Phenology Heat Map

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3))

ax.fill_between(das, daily["tmin"], daily["tmax"], alpha=0.3, color="tomato", label="Tmin–Tmax range")
ax.plot(das, (daily["tmax"] + daily["tmin"]) / 2, color="tomato", linewidth=1, label="Mean temperature")

# Shade growth stages
stage_colors = {1: "#d4f0d4", 2: "#a8d8a8", 3: "#7bc17b",
                4: "#f0e68c", 5: "#e0c060"}
for stage, color in stage_colors.items():
    mask = daily["istage"] == stage
    if mask.any():
        start = das[mask].iloc[0]
        end   = das[mask].iloc[-1]
        ax.axvspan(start, end, alpha=0.25, color=color,
                   label=f"Stage {stage}: {stage_labels.get(stage, '')}")

ax.set_xlabel("Days After Sowing")
ax.set_ylabel("Temperature (°C)")
ax.set_title("Temperature Range and Growth Stages")
ax.legend(fontsize=7, ncol=4, loc="upper right")
plt.tight_layout()
plt.show()

## 7  Using the DSSAT File Converters

The `dssat.io.converters` sub-package provides two-way conversion between legacy DSSAT
fixed-format files (`.WTH`, `.SOL`, X-files) and the JSON format used by this package.

### 7.1  Convert a `.WTH` file → JSON

In [ ]:
import tempfile, textwrap
from dssat.io.converters.wth import read_wth, write_wth

# Create a minimal in-memory WTH file (mimics a real DSSAT .WTH file)
WTH_TEXT = textwrap.dedent("""\
    $UFGA  Gainesville, FL
    @ INSI      LAT     LONG  ELEV   TAV   AMP REFHT WNDHT
      UFGA   29.630  -82.370   30.  22.5   7.8   2.0   2.0
    @DATE  SRAD  TMAX  TMIN  RAIN  WIND  RHUM
    82100  13.0  29.5  18.0   0.0  210.0  65.0
    82101  14.2  31.0  19.5   2.5  195.0  72.0
    82102  11.8  27.0  17.0   8.1  230.0  80.0
    82103  15.5  32.0  20.0   0.0  180.0  60.0
    82104  16.0  33.5  21.0   0.0  165.0  58.0
""")

with tempfile.NamedTemporaryFile(suffix=".WTH", mode="w", delete=False) as f:
    f.write(WTH_TEXT)
    wth_path = Path(f.name)

# Parse it
weather_data = read_wth(wth_path)

print("Station metadata:")
for k, v in weather_data.items():
    if k != "records":
        print(f"  {k:8s}: {v}")

print(f"\nDaily records ({len(weather_data['records'])} days):")
pd.DataFrame(weather_data["records"])

### 7.2  Convert JSON → `.WTH` (round-trip)

In [ ]:
with tempfile.NamedTemporaryFile(suffix=".WTH", mode="w", delete=False) as f:
    out_wth = Path(f.name)

write_wth(weather_data, out_wth, title="Round-trip test")

print("Written WTH file:")
print(out_wth.read_text())

### 7.3  Parse a `.SOL` soil file

In [ ]:
from dssat.io.converters.sol import read_sol

SOL_TEXT = textwrap.dedent("""\
    *IBMZ910014  IB  SIL   180  Millhopper Fine Sand, Gainesville FL
    @SITE        COUNTRY    LAT     LONG  SCS FAMILY
    Gainesville  USA        29.63  -82.37  Loamy, siliceous, hyperthermic
    @ SALB  SLPF  SMHB  SMPX  SMKE
      0.18  0.92   IB001 IB001 IB001
    @  SLB  SLLL  SDUL  SSAT  SSKS  SBDM  SLOC  SLCL  SLSI  SLHW  SRGF
         5 0.023 0.086 0.230  7.40  1.36  0.90   2.0  5.00   6.0  1.00
        15 0.023 0.086 0.230  7.40  1.40  0.69   2.0  5.00   6.0  0.85
        30 0.023 0.086 0.215  3.68  1.47  0.28   2.0  5.00   6.0  0.70
        60 0.025 0.087 0.185  0.88  1.66  0.09   3.0  5.00   6.3  0.50
        90 0.058 0.101 0.185  0.35  1.66  0.03   7.0  5.00   6.3  0.35
""")

with tempfile.NamedTemporaryFile(suffix=".SOL", mode="w", delete=False) as f:
    f.write(SOL_TEXT)
    sol_path = Path(f.name)

profiles = read_sol(sol_path)
profile  = profiles[0]

print("Profile ID  :", profile["id"])
print("Description :", profile.get("description"))
print(f"SALB={profile.get('salb')}  SLPF={profile.get('slpf')}")
print()
pd.DataFrame(profile["layers"])

## 8  Sensitivity: Varying Cultivar P1 Coefficient

A quick sensitivity analysis — how does the thermal time to end of juvenile phase
(P1, degree-days) affect grain yield and days to maturity?

In [ ]:
import copy, json

with open(EXPERIMENT_JSON) as f:
    base_exp = json.load(f)

p1_values = [160, 190, 220, 250, 280]  # degree-days

rows = []
for p1 in p1_values:
    exp_copy = copy.deepcopy(base_exp)
    exp_copy["crop"]["cultivar"]["p1"] = p1

    # Write modified experiment to a temp file so from_json can resolve the weather path
    with tempfile.NamedTemporaryFile(
        suffix=".json", mode="w", dir=TESTS_DIR, delete=False
    ) as f:
        json.dump(exp_copy, f)
        tmp_json = Path(f.name)

    try:
        sim_i = Simulation.from_json(tmp_json)
        res_i = sim_i.run()
        s = res_i["summary"]
        rows.append({
            "P1 (°C-d)": p1,
            "Yield (kg/ha)": round(s.get("yield_kg_ha", float("nan")), 1),
            "DAS to maturity": s.get("das_maturity", float("nan")),
            "Peak LAI": round(s.get("peak_lai", float("nan")), 2),
        })
    finally:
        tmp_json.unlink(missing_ok=True)

sens_df = pd.DataFrame(rows).set_index("P1 (°C-d)")
sens_df

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))

ax1.plot(p1_values, sens_df["Yield (kg/ha)"], marker="o", color="goldenrod")
ax1.set_xlabel("P1 (°C-d)")
ax1.set_ylabel("Grain Yield (kg ha⁻¹)")
ax1.set_title("P1 Sensitivity — Grain Yield")

ax2.plot(p1_values, sens_df["DAS to maturity"], marker="s", color="steelblue")
ax2.set_xlabel("P1 (°C-d)")
ax2.set_ylabel("Days to Maturity")
ax2.set_title("P1 Sensitivity — Days to Maturity")

plt.tight_layout()
plt.show()

## 9  Building an Experiment Programmatically

Instead of loading a JSON file you can assemble an experiment dict in Python and pass it directly
to `Simulation.from_json()` after writing it out — or construct the `Simulation` object manually.

In [ ]:
# Load base experiment and override a few fields programmatically
with open(EXPERIMENT_JSON) as f:
    new_exp = json.load(f)

# Change planting density and cultivar G2 (potential kernel number)
new_exp["planting"]["pltpop"] = 9.0        # plants m⁻²
new_exp["crop"]["cultivar"]["g2"] = 900.0  # kernels ear⁻¹

with tempfile.NamedTemporaryFile(
    suffix=".json", mode="w", dir=TESTS_DIR, delete=False
) as f:
    json.dump(new_exp, f)
    modified_path = Path(f.name)

try:
    sim_mod = Simulation.from_json(modified_path)
    res_mod = sim_mod.run()
    print("Modified experiment summary:")
    for k, v in res_mod["summary"].items():
        print(f"  {k:<30s}: {v}")
finally:
    modified_path.unlink(missing_ok=True)

---
## Summary

| Step | Code |
|------|------|
| Load experiment | `Simulation.from_json(path)` |
| Run simulation | `sim.run()` → `{"summary": dict, "daily": DataFrame}` |
| Convert WTH file | `read_wth(path)` / `write_wth(data, path)` |
| Convert SOL file | `read_sol(path)` / `write_sol(profiles, path)` |
| Convert X-file | `read_xfile(path)` / `write_xfile(exp, path)` |

The daily output DataFrame includes: `das`, `tmax`, `tmin`, `srad`, `rain`, `tsw_cm`,
`runoff_mm`, `drain_mm`, `snow_mm`, `istage`, `xstage`, `lai`, `biomas_g_m2`,
`yield_kg_ha`, `swfac`, `nstres`.
